# Invariance analysis

This notebook is separate from the study-specific notebooks and focuses only on invariance result analysis.

Use it to:

- scan `metric-results/` for invariance comparison JSON files
- flatten study-level metric deltas into one table
- inspect deltas and confidence intervals by study / metric
- visualise instability across Study A, Study B, Study B multi-turn, and Study C

Companion script:

- `scripts/evaluation/summarize_invariance_results.py`

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from reliable_clinical_benchmark.invariance_analysis import summarize_invariance_result_files

RUNTIME_ROOT = Path.cwd().resolve().parents[0]
RESULTS_ROOT = RUNTIME_ROOT / "metric-results"
MANIFEST_ROOT = RUNTIME_ROOT / "data" / "frozen_splits" / "v5_invariance_samples"

rows = summarize_invariance_result_files(RESULTS_ROOT)
df = pd.DataFrame(rows)
if df.empty:
    print("No invariance comparison JSON files found under", RESULTS_ROOT)
else:
    print(f"Loaded {len(df)} invariance metric rows from {RESULTS_ROOT}")
    display(df.head())

In [ ]:
if not df.empty:
    summary = (
        df.sort_values(["study", "metric", "variant_cache"])
        [["study", "metric", "n_pairs", "base", "variant", "delta", "ci_low", "ci_high"]]
        .reset_index(drop=True)
    )
    display(summary)

    manifest_counts = json.loads((MANIFEST_ROOT / "manifest.json").read_text(encoding="utf-8"))
    display(pd.DataFrame.from_dict(manifest_counts["studies"], orient="index"))

In [ ]:
def plot_invariance_deltas(frame: pd.DataFrame, study_name: str) -> None:
    subset = frame[frame["study"] == study_name].copy()
    if subset.empty:
        print(f"No invariance rows for {study_name}")
        return

    subset = subset.sort_values("metric")
    y = range(len(subset))
    lower = subset["delta"] - subset["ci_low"]
    upper = subset["ci_high"] - subset["delta"]

    plt.figure(figsize=(10, max(3, len(subset) * 0.6)))
    plt.errorbar(subset["delta"], y, xerr=[lower, upper], fmt="o")
    plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
    plt.yticks(list(y), subset["metric"])
    plt.title(f"Invariance deltas: {study_name}")
    plt.xlabel("variant - base")
    plt.tight_layout()
    plt.show()


if not df.empty:
    for study_name in sorted(df["study"].unique()):
        plot_invariance_deltas(df, study_name)

In [ ]:
if not df.empty:
    flagged = df[(df["ci_low"] > 0) | (df["ci_high"] < 0)].copy()
    flagged = flagged.sort_values(["study", "metric", "delta"], ascending=[True, True, False])
    display(flagged[["study", "metric", "delta", "ci_low", "ci_high", "variant_cache"]])

    by_study = (
        df.groupby("study", as_index=False)
        .agg(mean_abs_delta=("delta", lambda s: float(s.abs().mean())), n_metrics=("metric", "count"))
        .sort_values("mean_abs_delta", ascending=False)
    )
    display(by_study)